In [ ]:
# Cell 1: config + browser startup.
import importlib
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import browser.driver as browser_driver
import core.config as core_config
import parsers.page_markdown as page_markdown
import browser.markdown as markdown_tools
import importlib as _importlib
import browser.webagent as browser_webagent
browser_interact = _importlib.import_module("browser.interact")
import tests.llm_test.llm_runtime as llm_runtime

browser_driver = importlib.reload(browser_driver)
core_config = importlib.reload(core_config)
page_markdown = importlib.reload(page_markdown)
markdown_tools = importlib.reload(markdown_tools)
browser_webagent = importlib.reload(browser_webagent)
browser_interact = _importlib.reload(browser_interact)
llm_runtime = importlib.reload(llm_runtime)

from browser.driver import build_driver
from core.config import load_app_config
from storage.embedded_mongo import EmbeddedMongoStore

app_config = load_app_config()
llm_config = app_config["llm_backend"]
browser_cfg = dict(app_config["profile"]["browser"])
old_user_data_dir = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
if old_user_data_dir.exists():
    browser_cfg["user_data_dir"] = str(old_user_data_dir)
openai_api_key_path = Path(r"D:\_Desktop\api_key.txt")
store = EmbeddedMongoStore(Path(app_config["profile"]["mongo_file"]))
driver = build_driver(browser_cfg)
LLM_TEST_DIR = ROOT / "tests" / "llm_test"
markdown_text = ""
interactables = {
    "interactables": [],
    "images": [],
    "rows": [],
    "counts": {"interactables": 0, "images": 0, "rows": 0},
}
session_outputs = []
runtime_state = {
    "driver": driver,
    "store": store,
    "log_path": "mock://mongodb/webagent",
    "llm_backend": llm_config,
    "model_backend": llm_config["backend"],
    "api_key_path": str(Path(llm_config[llm_config["backend"]].get("api_key_path", openai_api_key_path))),
    "openai_model": llm_config.get("openai", {}).get("model", "gpt-5.4-mini"),
    "openai_base_url": llm_config.get("openai", {}).get("base_url", "https://api.openai.com/v1/chat/completions"),
    "openai_reasoning_effort": llm_config.get("openai", {}).get("reasoning_effort", "low"),
    "llama_model": llm_config.get("local", {}).get("model", "qwen3.5-9b"),
    "llama_base_url": llm_config.get("local", {}).get("base_url", "http://127.0.0.1:8080/v1/chat/completions"),
    "markdown_text": markdown_text,
    "interactables": interactables,
    "session_outputs": session_outputs,
    "delays": {
        "webagent_click": 2,
        "webagent_type": 2,
        "webagent_clear_text": 2
    },
    "context_tokens": 20000,
    "response_reserve_tokens": 2048,
    "verbose": True,
    "wait_seconds": 0,
}

print(json.dumps({"root": str(ROOT), "browser": browser_cfg, "old_user_data_dir": str(old_user_data_dir)}, indent=2))


In [ ]:
# Cell 2: runtime loop.
import json
from typing import Any

from tests.llm_test.llm_runtime import (
    append_transcript_text,
    build_messages,
    call_model_chat_with_retry,
    display_block,
    display_json,
    execute_tool_call,
    format_tool_result_for_display,
    format_tool_result_for_llm,
    _tool_target_text,
    load_instruction_bundle,
    trim_messages_to_budget,
    parse_model_output,
    stop_requested,
)

INSTRUCTIONS = load_instruction_bundle(LLM_TEST_DIR)
llm_config = runtime_state["llm_backend"]
MODEL_BACKEND = llm_config["backend"]
OPENAI_MODEL = llm_config.get("openai", {}).get("model", "gpt-5.4-mini")
OPENAI_API_BASE = llm_config.get("openai", {}).get("base_url", "https://api.openai.com/v1/chat/completions")
OPENAI_REASONING_EFFORT = llm_config.get("openai", {}).get("reasoning_effort", "low")
LLAMA_MODEL = llm_config.get("local", {}).get("model", "qwen3.5-9b")
LLAMA_API_BASE = llm_config.get("local", {}).get("base_url", "http://127.0.0.1:8080/v1/chat/completions")


def run_agent(task: str, max_steps: int = 20) -> dict[str, Any]:
    runtime = runtime_state
    runtime["task"] = task
    runtime["session_outputs"] = session_outputs
    runtime["stuck_counts"] = {}

    display_block("User task", task, kind="user")
    messages = build_messages(task, INSTRUCTIONS, include_plan=True)
    base_message_count = len(messages)
    context_tokens = int(runtime.get("context_tokens", 20000))
    response_reserve_tokens = int(runtime.get("response_reserve_tokens", 2048))

    for step in range(max_steps):
        if stop_requested():
            final_text = "Loop halted by ] key."
            display_block("Final response", final_text, kind="final")
            runtime["last_result"] = {"kind": "halted", "text": final_text}
            return runtime["last_result"]

        messages = trim_messages_to_budget(
            messages,
            max_context_tokens=context_tokens,
            response_reserve_tokens=response_reserve_tokens,
            keep_head=base_message_count,
        )

        try:
            llm_text = call_model_chat_with_retry(
                MODEL_BACKEND,
                messages=messages,
                api_key_path=runtime["api_key_path"],
                openai_model=OPENAI_MODEL,
                openai_base_url=OPENAI_API_BASE,
                openai_reasoning_effort=OPENAI_REASONING_EFFORT,
                llama_model=LLAMA_MODEL,
                llama_base_url=LLAMA_API_BASE,
                max_completion_tokens=1024,
                timeout=300,
                retries=2,
            )
        except Exception as exc:
            error_text = str(exc)
            display_block(f"LLM error {step + 1}", error_text, kind="error")
            runtime["last_result"] = {"kind": "error", "message": error_text}
            return runtime["last_result"]
        display_block(f"LLM response {step + 1}", llm_text, kind="llm")
        messages.append({"role": "assistant", "content": llm_text})
        messages = trim_messages_to_budget(
            messages,
            max_context_tokens=context_tokens,
            response_reserve_tokens=response_reserve_tokens,
            keep_head=base_message_count,
        )
        append_transcript_text(runtime, "llm", f"step-{step + 1}", llm_text)

        parsed = parse_model_output(llm_text)
        if parsed["kind"] == "final_response":
            display_block("Final response", parsed["text"], kind="final")
            runtime["last_result"] = parsed
            append_transcript_text(runtime, "final", "response", parsed["text"])
            return parsed

        if parsed["kind"] == "error":
            display_json("Parsed error", parsed, kind="error")
            messages.append(
                {
                    "role": "user",
                    "content": "The previous response was invalid. Return exactly one <cmd>...</cmd> or one <final_response>...</final_response>.",
                }
            )
            messages = trim_messages_to_budget(
                messages,
                max_context_tokens=context_tokens,
                response_reserve_tokens=response_reserve_tokens,
                keep_head=base_message_count,
            )
            key = parsed.get("message", "parse-error")
            runtime["stuck_counts"][key] = runtime["stuck_counts"].get(key, 0) + 1
            if runtime["stuck_counts"][key] > 4:
                final_text = f"Stopped after repeated parse errors: {key}"
                display_block("Final response", final_text, kind="final")
                runtime["last_result"] = {"kind": "stopped", "text": final_text}
                return runtime["last_result"]
            continue

        if parsed["kind"] == "text":
            display_block(f"Plan or text {step + 1}", parsed["text"], kind="plan")
            messages.append(
                {
                    "role": "user",
                    "content": "Now return exactly one tool call in <cmd>...</cmd> format, or stop with <final_response>...</final_response>.",
                }
            )
            messages = trim_messages_to_budget(
                messages,
                max_context_tokens=context_tokens,
                response_reserve_tokens=response_reserve_tokens,
                keep_head=base_message_count,
            )
            append_transcript_text(runtime, "plan", f"step-{step + 1}", parsed["text"])
            continue

        if parsed["kind"] == "command":
            cmd_text = f'<cmd>{parsed["raw"]}</cmd>'
            display_block(f"Parsed cmd {step + 1}", cmd_text, kind="cmd")
            result = execute_tool_call(parsed["tool"], parsed["args"], runtime)
            output_text = format_tool_result_for_llm(result, runtime)
            target_text = _tool_target_text(result)
            output_label = f"Tool output {step + 1}"
            if target_text:
                output_label += f" - {target_text}"
            display_block(output_label, format_tool_result_for_display(result, runtime), kind="output")
            append_transcript_text(runtime, "cmd", f"step-{step + 1}", cmd_text)
            append_transcript_text(runtime, "output", f"step-{step + 1}", result)
            messages.append({"role": "user", "content": cmd_text + "\n<output>" + output_text + "</output>"})
            messages = trim_messages_to_budget(
                messages,
                max_context_tokens=context_tokens,
                response_reserve_tokens=response_reserve_tokens,
                keep_head=base_message_count,
            )

            stuck_key = f'{parsed["tool"]}:{result.get("status", "unknown")}:{result.get("message", "")}'
            runtime["stuck_counts"][stuck_key] = runtime["stuck_counts"].get(stuck_key, 0) + 1
            if runtime["stuck_counts"][stuck_key] > 4:
                final_text = f'Stopped after repeating the same tool state: {stuck_key}'
                display_block("Final response", final_text, kind="final")
                runtime["last_result"] = {"kind": "stopped", "text": final_text, "last_tool": parsed["tool"]}
                return runtime["last_result"]
            continue

        display_json("Unexpected parse result", parsed, kind="error")
        messages.append(
            {
                "role": "user",
                "content": "Return a plan, a single <cmd>...</cmd>, or a <final_response>...</final_response>.",
            }
        )

    final_text = f"Stopped after max_steps={max_steps}."
    display_block("Final response", final_text, kind="final")
    runtime["last_result"] = {"kind": "stopped", "text": final_text}
    return runtime["last_result"]


# Example call.
# run_agent("Open the page and inspect it with webuse tools.")


In [ ]:
# Cell 3: run a task example.
# Edit the task string below, then run this cell.

task = "Go to DuckDuckGo and search for the cheapest VPS server available."
result = run_agent(task)
result
